# YLM v1: run training packs on Colab

1. Runtime -> Change runtime type -> **GPU** (T4 is fine).
2. Put `ylm_data.zip` (made by `colab/make_bundle.sh`; the data files are not in git) into **MyDrive/ylm/** on Google Drive.
3. Run the cells in order. The code is cloned from GitHub (branch below). Pick one `PACK`; each line of a pack file is one training.
   Results are copied to **MyDrive/ylm/runs/** every 5 minutes, so a disconnect does not lose finished runs
   (a run that was cut off is lost: re-run the pack).

Packs: `v1_0 ... v1_3` = main interface table (seeds 0-2: bc, optA, prog, rule, oracle), `v1_4 ... v1_7` = ablations,
`pol` = polarizability (3 seeds), `quad_s0..2` = l=2 quadrupole. Use a different pack in each Colab tab/account.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, zipfile
BRANCH = 'exp/v1-benchmark'
DATA = '/content/drive/MyDrive/ylm/ylm_data.zip'
assert os.path.exists(DATA), 'upload ylm_data.zip to MyDrive/ylm/ first'
if not os.path.exists('/content/YLM'):
    !git clone --depth 1 --branch {BRANCH} https://github.com/yaniguan/YLM.git /content/YLM
os.chdir('/content/YLM')
!git log --oneline | head -1
os.makedirs('data', exist_ok=True)
with zipfile.ZipFile(DATA) as z: z.extractall('.')
!pip -q install "e3nn==0.6.0" scipy
!python -m pytest tests -q 2>&1 | tail -3

In [ ]:
PACK = 'jobs/packs/v1_0.txt'     # <- choose
N_PARALLEL = 3                    # 3 concurrent trainings fit a 16 GB T4
EXTRA = ''                        # e.g. '--epochs 20' (results are then NOT comparable with the cluster runs)
print(open(PACK).read())

In [ ]:
import threading, time, subprocess
os.makedirs('runs', exist_ok=True)
DRIVE_RUNS = '/content/drive/MyDrive/ylm/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
stop = False
def sync():
    while not stop:
        subprocess.run(['rsync', '-a', 'runs/', DRIVE_RUNS + '/'])
        time.sleep(300)
threading.Thread(target=sync, daemon=True).start()

cmds = [l.strip() for l in open(PACK) if l.strip()]
cmds = [c.replace(' > ', ' ' + EXTRA + ' > ', 1) if EXTRA else c for c in cmds]
pending, running = list(cmds), []
t0 = time.time()
while pending or running:
    running = [p for p in running if p.poll() is None]
    while pending and len(running) < N_PARALLEL:
        c = pending.pop(0); print(time.strftime('%H:%M:%S'), 'start', c[:110], flush=True)
        running.append(subprocess.Popen(c, shell=True)); time.sleep(20)
    time.sleep(30)
stop = True
subprocess.run(['rsync', '-a', 'runs/', DRIVE_RUNS + '/'])
print('all done in %.1f h' % ((time.time() - t0) / 3600))

In [ ]:
# progress of every run (last epoch line)
import glob
for f in sorted(glob.glob('runs/*.stdout')):
    lines = [l for l in open(f) if l.startswith('epoch') or 'Error' in l]
    print(os.path.basename(f), '|', (lines[-1].strip()[:150] if lines else 'no epoch yet'))